# 03 - Sequence Baselines (Leakage-Safe Split)

Evaluates unigram and bigram next-token baselines using file-level train/val/test splits from notebook 02.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from sklearn.metrics import top_k_accuracy_score

PROJECT_OUTPUT_DIR = Path('/content/project_outputs')
CACHE_DIR = PROJECT_OUTPUT_DIR / 'cache'
TABLE_DIR = PROJECT_OUTPUT_DIR / 'tables'

with open(CACHE_DIR / 'sequence_cache.json', 'r') as f:
    cache = json.load(f)

V = len(cache['vocab'])
sw = cache['split_windows']


In [ ]:
def eval_unigram(train_y, test_y, V):
    cnt = Counter(train_y)
    total = sum(cnt.values())
    probs = np.array([cnt.get(i,0)/max(total,1) for i in range(V)])
    pred = int(np.argmax(probs))
    acc = np.mean(np.array(test_y) == pred)
    p_true = np.array([max(probs[t], 1e-12) for t in test_y])
    ce = -np.mean(np.log(p_true))
    ppl = float(np.exp(ce))
    top5 = top_k_accuracy_score(test_y, np.tile(probs, (len(test_y),1)), k=min(5,V), labels=np.arange(V))
    return {'accuracy': float(acc), 'cross_entropy': float(ce), 'perplexity': ppl, 'top5': float(top5)}


def eval_bigram(train_X, train_y, test_X, test_y, V):
    trans = defaultdict(Counter)
    for x, y in zip(train_X, train_y):
        trans[x[-1]][y] += 1

    global_cnt = Counter(train_y)
    gtot = sum(global_cnt.values())
    gprob = np.array([global_cnt.get(i,0)/max(gtot,1) for i in range(V)])

    probs_rows, preds = [], []
    for x in test_X:
        cnt = trans.get(x[-1])
        if not cnt:
            probs = gprob
        else:
            tot = sum(cnt.values())
            probs = np.array([cnt.get(i,0)/max(tot,1) for i in range(V)])
        probs_rows.append(probs)
        preds.append(int(np.argmax(probs)))

    probs_arr = np.vstack(probs_rows)
    acc = np.mean(np.array(test_y) == np.array(preds))
    p_true = np.array([max(probs_arr[i, t], 1e-12) for i, t in enumerate(test_y)])
    ce = -np.mean(np.log(p_true))
    ppl = float(np.exp(ce))
    top5 = top_k_accuracy_score(test_y, probs_arr, k=min(5,V), labels=np.arange(V))
    return {'accuracy': float(acc), 'cross_entropy': float(ce), 'perplexity': ppl, 'top5': float(top5)}


In [ ]:
rows = []
for name, key in [('raw_time','raw'), ('quantized_time','quant')]:
    trX, trY = sw[key]['train_X'], sw[key]['train_y']
    teX, teY = sw[key]['test_X'], sw[key]['test_y']

    u = eval_unigram(trY, teY, V)
    rows.append({'setting': name, 'model': 'unigram', **u})

    b = eval_bigram(trX, trY, teX, teY, V)
    rows.append({'setting': name, 'model': 'bigram_markov', **b})

baseline_df = pd.DataFrame(rows)
display(baseline_df)
baseline_df.to_csv(TABLE_DIR / '03_baseline_results.csv', index=False)
print('Saved notebook 03 outputs.')
